# BIT-on-DOFA — Integration Pipeline (clean)

Replaces BIT's ImageNet ResNet backbone with the **DOFA** Earth-observation foundation model + a shape adapter.

**Run top-to-bottom on a GPU runtime** (Runtime → Change runtime type → T4 GPU).

## 1. Setup — clone DOFA & download pretrained weights

In [ ]:
!git clone https://github.com/zhu-xlab/DOFA.git
%cd DOFA
%run checkpoints/download_weights.py

## 2. Imports & load the DOFA encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dofa_v1 import vit_base_patch16

check_point = torch.load('./checkpoints/DOFA_ViT_base_e100.pth')
vit_model = vit_base_patch16()
vit_model.load_state_dict(check_point, strict=False)   # strict=False: we discard DOFA's head
vit_model = vit_model.cuda()
print('DOFA loaded')

## 3. DOFA → spatial patch tokens

Returns DOFA's per-patch features `[B, 196, 768]` — the spatial grid, *before* the global pooling that `forward_features` does.

*(TODO before training: add `model.fc_norm` back for properly-normalized feature values — dropped here, doesn't affect shape.)*

In [ ]:
def get_spatial_tokens(model, x, wave_list):
    waves = torch.tensor(wave_list, device=x.device).float()
    model.waves = waves
    x, _ = model.patch_embed(x, model.waves)                       # image -> patch tokens
    x = x + model.pos_embed[:, 1:, :]                              # add position info
    cls = (model.cls_token + model.pos_embed[:, :1, :]).expand(x.shape[0], -1, -1)
    x = torch.cat((cls, x), dim=1)                                # prepend cls token
    for block in model.blocks:                                    # transformer blocks
        x = block(x)
    return x[:, 1:, :]                                            # patch tokens only -> [B, 196, 768]

## 4. The adapter — drop-in ResNet replacement

Converts DOFA's output into the **exact** shape BIT's backbone produces: `[B, 32, 64, 64]`.
- `squeeze`: depth 768 → 32 (a 1×1 conv, learnable)
- `interpolate`: size 14×14 → 64×64
- also shrinks the 256 input → 224 (DOFA was pretrained at 224).

In [ ]:
squeeze = nn.Conv2d(768, 32, kernel_size=1).cuda()   # depth fixer: 768 -> 32

def dofa_backbone(image, model, squeeze, wave_list=[0.665, 0.56, 0.49]):
    image = F.interpolate(image, size=(224, 224), mode='bilinear', align_corners=False)  # 256 -> 224
    tokens = get_spatial_tokens(model, image, wave_list)              # [B, 196, 768]
    B = image.shape[0]
    grid = tokens.reshape(B, 14, 14, 768).permute(0, 3, 1, 2)         # [B, 768, 14, 14]
    grid = squeeze(grid)                                             # [B, 32, 14, 14]
    grid = F.interpolate(grid, size=(64, 64), mode='bilinear', align_corners=False)  # [B, 32, 64, 64]
    return grid

## 5. Build BIT & swap the backbone to DOFA

In [ ]:
!git clone https://github.com/rushammm/BIT_CD.git /content/BIT_CD
import sys, types
sys.path.insert(0, '/content/BIT_CD')
from models.networks import BASE_Transformer

# build the exact base_transformer_pos_s4_dd8 config
bit = BASE_Transformer(input_nc=3, output_nc=2, token_len=4, resnet_stages_num=4,
                       with_pos='learned', enc_depth=1, dec_depth=8).cuda()

# attach DOFA + adapter, freeze DOFA, swap forward_single
bit.dofa = vit_model
bit.squeeze = squeeze
for p in bit.dofa.parameters():
    p.requires_grad = False
bit.forward_single = types.MethodType(lambda self, x: dofa_backbone(x, self.dofa, self.squeeze), bit)
print('BIT-on-DOFA ready')

## 6. Test end-to-end

Expect `torch.Size([1, 2, 256, 256])` — a full change map, no errors.

In [ ]:
A = torch.randn(1, 3, 256, 256).cuda()   # fake 'before' image
B = torch.randn(1, 3, 256, 256).cuda()   # fake 'after' image
out = bit(A, B)
print(out.shape)